# MIA Visualization

In [1]:
import os
from pathlib import Path
from typing import Any

import ipywidgets as widgets
import plotly.express as px
from IPython.display import display
from plotly.subplots import make_subplots
import plotly.graph_objects as go

In [2]:
FIG_HEIGHT = 800
FIG_WIDTH = 1100
FONT_SIZE = 32

## Utilities

In [3]:
def format_metric_name(metric_name: str) -> str:
    """
    Prettify diagram texts by replacing
    snake case with regular title case.
    """
    output_words = []
    for word in metric_name.split("_"):
        word = word.title() if word not in ["FPR", "TPR"] else word.upper()
        output_words.append(word)

    return " ".join(output_words)

## Plotting

In [4]:
repo_abs_path = Path(os.path.abspath("")).parent.parent

plots_dir = f"{repo_abs_path}/examples/visualizations/tf_shadow_data_variations"

In [5]:
def customize_figure_layout(gen_fig: Any, render_legend: bool = True) -> None:
    if render_legend:
        gen_fig.update_layout(
            height=FIG_HEIGHT,
            width=FIG_WIDTH,
            font_color="black",
            legend=dict(
                y=1.0,
                x=0.5,
                xanchor="center",
                yanchor="bottom",
                orientation="h",
                valign="top",
                title_text="",
                font=dict(size=FONT_SIZE+4),
                title_font_family="Helvetica",
            ),
            title_font_family="Helvetica",
            title_x=0.5,
            title_y=0.975,
            margin=dict(l=30, r=30, t=75, b=25),
            plot_bgcolor="white",
            font=dict(size=FONT_SIZE+10, family="Helvetica"),
        )
    else:
        gen_fig.update_layout(
            height=FIG_HEIGHT,
            width=FIG_WIDTH,
            font_color="black",
            legend={},
            title_font_family="Helvetica",
            title_x=0.5,
            title_y=0.975,
            margin=dict(l=30, r=30, t=75, b=25),
            plot_bgcolor="white",
            font=dict(size=FONT_SIZE + 10, family="Helvetica"),
        )

def trim_png_whitespace(image_path: str, pad: int = 2) -> None:
    """Crop near-white borders from a saved PNG."""
    import numpy as np
    from PIL import Image

    img = Image.open(image_path)
    arr = np.asarray(img)
    content = np.any(arr[:, :, :3] < 250, axis=2) if arr.ndim == 3 else arr < 250
    rows = np.any(content, axis=1)
    cols = np.any(content, axis=0)
    if not rows.any() or not cols.any():
        return

    rmin, rmax = np.where(rows)[0][[0, -1]]
    cmin, cmax = np.where(cols)[0][[0, -1]]
    rmin = max(0, int(rmin) - pad)
    cmin = max(0, int(cmin) - pad)
    rmax = min(arr.shape[0] - 1, int(rmax) + pad)
    cmax = min(arr.shape[1] - 1, int(cmax) + pad)
    img.crop((cmin, rmin, cmax + 1, rmax + 1)).save(image_path)


def save_and_display_figure(gen_fig: Any, template_name: str, plots_dir: str) -> None:
    plot_png_path = f"{plots_dir}/{template_name}.png"
    plot_pdf_path = f"{plots_dir}/{template_name}.pdf"

    os.makedirs(plots_dir, exist_ok=True)
    gen_fig.write_image(plot_png_path, scale=2)
    gen_fig.write_image(plot_pdf_path)
    trim_png_whitespace(plot_png_path)

    display(gen_fig)

In [6]:
mismatched_specifics = ["20K", "50K", "100K", "200K"]
mismatch_type = ["20 Shadows", "20 Shadows", "20 Shadows", "20 Shadows"]
mismatch_mia = [0.449, 0.444, 0.418, 0.405]
df_1 = {"mismatched_specifics": mismatched_specifics, "Mismatch Type": mismatch_type, "mismatch_mia": mismatch_mia}

mismatched_specifics_2 = ["20K", "50K", "100K", "200K"]
mismatch_type_2 = ["1 Shadow", "1 Shadow", "1 Shadow", "1 Shadow"]
mismatch_mia_2 = [0.429, 0.417, 0.395, 0.310]
df_2 = {"mismatched_specifics": mismatched_specifics_2, "Mismatch Type": mismatch_type_2, "mismatch_mia": mismatch_mia_2}

fig = go.Figure()

fig1 = px.bar(
    data_frame=df_1,
    x="mismatched_specifics",
    y="mismatch_mia",
    color="mismatched_specifics",
    color_discrete_sequence=["#53abff", "#f78d8d", "#660066", "#06c2ac"],
)

fig2 = px.bar(
    data_frame=df_2,
    x="mismatched_specifics",
    y="mismatch_mia",
    color="mismatched_specifics",
    color_discrete_sequence=["#53abff", "#f78d8d", "#660066", "#06c2ac"],
)

fig = make_subplots(rows=1, cols=2, shared_yaxes=True, horizontal_spacing=0.08)

for trace in fig1.data:
    fig.add_trace(trace, row=1, col=1)

for trace in fig2.data:
    fig.add_trace(trace, row=1, col=2)

fig.update_layout(bargap=0.0, showlegend=False)
fig.update_xaxes(
  ticks="outside", tickwidth=4, ticklen=7.5, linewidth=4, linecolor="black", tickangle=-45, title_font_family="Helvetica", tickfont_family="Helvetica", tickfont_size=FONT_SIZE+10,
)
fig.update_yaxes(
    ticks="outside", tickwidth=4, ticklen=7.5, linewidth=4, linecolor="black", title = "TF (WB) Success", tickangle=0, title_font_family="Helvetica", tickfont_family="Helvetica", tickfont_size=FONT_SIZE+10,
)
fig.update_yaxes(tickmode='array', tickvals=[0.1, 0.2, 0.3, 0.4, 0.5], range=[0.1, 0.5])
fig.update_traces(marker_line_width=4, marker_line_color="black")
fig.update_yaxes(showticklabels=False, title_text=None, row=1, col=2)
fig.update_xaxes(title_text="20 Shadows", categoryorder="array", categoryarray=["20K", "50K", "100K", "200K"], row=1, col=1)
fig.update_xaxes(title_text="1 Shadow", categoryorder="array", categoryarray=["20K", "50K", "100K", "200K"], row=1, col=2)
fig.update_yaxes(visible=False, row=1, col=2)

customize_figure_layout(fig, render_legend=False)
save_and_display_figure(fig, "berka_tf_shadow_data_variations", plots_dir)

In [7]:
mismatched_specifics = ["10K", "15K", "20K", "25K"]
mismatch_type = ["3 Shadows", "3 Shadows", "3 Shadows", "3 Shadows"]
mismatch_mia = [0.543, 0.510, 0.472, 0.465]
df_1 = {"mismatched_specifics": mismatched_specifics, "Mismatch Type": mismatch_type, "mismatch_mia": mismatch_mia}

mismatched_specifics_2 = ["10K", "15K", "20K", "25K"]
mismatch_type_2 = ["1 Shadow", "1 Shadow", "1 Shadow", "1 Shadow"]
mismatch_mia_2 = [0.536, 0.453, 0.443, 0.441]
df_2 = {"mismatched_specifics": mismatched_specifics_2, "Mismatch Type": mismatch_type_2, "mismatch_mia": mismatch_mia_2}

fig = go.Figure()

fig1 = px.bar(
    data_frame=df_1,
    x="mismatched_specifics",
    y="mismatch_mia",
    color="mismatched_specifics",
    color_discrete_sequence=["#53abff", "#f78d8d", "#660066", "#06c2ac"],
)

fig2 = px.bar(
    data_frame=df_2,
    x="mismatched_specifics",
    y="mismatch_mia",
    color="mismatched_specifics",
    color_discrete_sequence=["#53abff", "#f78d8d", "#660066", "#06c2ac"],
)

fig = make_subplots(rows=1, cols=2, shared_yaxes=True, horizontal_spacing=0.08)

for trace in fig1.data:
    fig.add_trace(trace, row=1, col=1)

for trace in fig2.data:
    fig.add_trace(trace, row=1, col=2)

fig.update_layout(bargap=0.0, showlegend=False)
fig.update_xaxes(
  ticks="outside", tickwidth=4, ticklen=7.5, linewidth=4, linecolor="black", tickangle=-45, title_font_family="Helvetica", tickfont_family="Helvetica", tickfont_size=FONT_SIZE+10,
)
fig.update_yaxes(
    ticks="outside", tickwidth=4, ticklen=7.5, linewidth=4, linecolor="black", title = "TF (WB) Success", tickangle=0, title_font_family="Helvetica", tickfont_family="Helvetica", tickfont_size=FONT_SIZE+10,
)
fig.update_yaxes(tickmode='array', tickvals=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6], range=[0.1, 0.6])
fig.update_traces(marker_line_width=4, marker_line_color="black")
fig.update_yaxes(showticklabels=False, title_text=None, row=1, col=2)
fig.update_xaxes(title_text="3 Shadows", categoryorder="array", categoryarray=["10K", "15K", "20K", "25K"], row=1, col=1)
fig.update_xaxes(title_text="1 Shadow", categoryorder="array", categoryarray=["10K", "15K", "20K", "25K"], row=1, col=2)
fig.update_yaxes(visible=False, row=1, col=2)

customize_figure_layout(fig, render_legend=False)
save_and_display_figure(fig, "diabetes_tf_shadow_data_variations", plots_dir)

In [8]:
# DOMIAS Reference data variations

mismatched_specifics = ["800K", "600K", "400K", "100K", "50K", "None"]
mismatch_type = ["Berka", "Berka", "Berka", "Berka", "Berka", "Berka"]
mismatch_mia = [0.215, 0.207, 0.187, 0.187, 0.188, 0.182]
df_1 = {"mismatched_specifics": mismatched_specifics, "Mismatch Type": mismatch_type, "mismatch_mia": mismatch_mia}

mismatched_specifics_2 = ["70K", "50K", "35K", "10K", "None"]
mismatch_type_2 = ["Diabetes", "Diabetes", "Diabetes", "Diabetes", "Diabetes"]
mismatch_mia_2 = [0.281, 0.280, 0.286, 0.293, 0.297]
df_2 = {"mismatched_specifics": mismatched_specifics_2, "Mismatch Type": mismatch_type_2, "mismatch_mia": mismatch_mia_2}

fig = go.Figure()

fig1 = px.bar(
    data_frame=df_1,
    x="mismatched_specifics",
    y="mismatch_mia",
    color="mismatched_specifics",
    color_discrete_sequence=["#53abff", "#f78d8d", "#660066", "#06c2ac", "#ff796c", "#000000"],
)

fig2 = px.bar(
    data_frame=df_2,
    x="mismatched_specifics",
    y="mismatch_mia",
    color="mismatched_specifics",
    color_discrete_sequence=["#53abff", "#f78d8d", "#660066", "#06c2ac", "#000000"],
)

fig = make_subplots(rows=1, cols=2, shared_yaxes=True, horizontal_spacing=0.08)

for trace in fig1.data:
    fig.add_trace(trace, row=1, col=1)

for trace in fig2.data:
    fig.add_trace(trace, row=1, col=2)

fig.update_layout(bargap=0.0, showlegend=False)
fig.update_xaxes(
  ticks="outside", tickwidth=4, ticklen=7.5, linewidth=4, linecolor="black", tickangle=-45, title_font_family="Helvetica", tickfont_family="Helvetica", tickfont_size=FONT_SIZE+10,
)
fig.update_yaxes(
    ticks="outside", tickwidth=4, ticklen=7.5, linewidth=4, linecolor="black", title = "Ensemble (BB) Success", tickangle=0, title_font_family="Helvetica", tickfont_family="Helvetica", tickfont_size=FONT_SIZE+10,
)
fig.update_yaxes(tickmode='array', tickvals=[0.0, 0.1, 0.2, 0.3], range=[0.0, 0.3])
fig.update_traces(marker_line_width=4, marker_line_color="black")
fig.update_yaxes(showticklabels=False, title_text=None, row=1, col=2)
fig.update_xaxes(title_text="Berka", categoryorder="array", categoryarray=["800K", "600K", "400K", "100K", "50K", "None"], row=1, col=1)
fig.update_xaxes(title_text="Diabetes", categoryorder="array", categoryarray=["70K", "50K", "35K", "10K", "None"], row=1, col=2)
fig.update_yaxes(visible=False, row=1, col=2)

customize_figure_layout(fig, render_legend=False)
save_and_display_figure(fig, "ensemble_domias_reference_data_variations", plots_dir)